# Módulo 1: Fundamentos de IA y Regresión
### Workshop: IA aplicada a Distribución de Energía
**Presentado por: Dr. Francisco Arduh - Xcapit**

---

## Agenda (~2 horas)

| Bloque | Tema | Duración |
|--------|------|----------|
| 0 | Bienvenida, Colab y Python básico | ~15 min |
| 1 | El paisaje de la IA (conceptual, sin código) | ~15 min |
| 2 | Anatomía de un modelo de ML | ~20 min |
| 3 | Caso práctico: Predicción de demanda (Regresión) | ~50 min |
| 4 | Teaser: Clasificación (detección de anomalías) | ~20 min |

---
## Sección 0: Bienvenida y Setup

### Cómo usar este notebook

Estamos en **Google Colab**, un entorno gratuito que permite ejecutar código Python en la nube. No necesitan instalar nada.

- Cada "bloque" se llama **celda**. Hay celdas de texto (como esta) y celdas de código.
- Para ejecutar una celda de código: **Shift + Enter** (o el botón de Play).
- Las celdas se ejecutan **en orden**. Si saltean una, puede fallar la siguiente.
- Si algo se rompe: **Menú > Entorno de ejecución > Reiniciar entorno** y ejecutar todo de nuevo.

In [ ]:
# Esta es una celda de codigo. Ejecutenla con Shift+Enter.
print("Hola! Si ven este mensaje, Colab funciona correctamente.")

In [ ]:
# Importamos las librerias que vamos a usar en todo el modulo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
print("Librerias cargadas correctamente.")

### Python en 5 minutos (para gente que sabe SQL)

No vamos a aprender Python completo. Solo lo mínimo para seguir el workshop.

**Idea clave:** En Python trabajamos con **DataFrames** (librería `pandas`), que son básicamente **tablas**, como las de SQL.

In [ ]:
# Variables: como declarar valores
temperatura = 32.5
zona = "Norte"
es_feriado = True

print(f"Temperatura: {temperatura}, Zona: {zona}, Feriado: {es_feriado}")

In [ ]:
# Un DataFrame es como una tabla SQL
clientes = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Bruno', 'Carla', 'Diego', 'Elena'],
    'consumo_kwh': [350, 120, 890, 450, 200],
    'zona': ['Norte', 'Sur', 'Norte', 'Centro', 'Sur']
})

clientes  # Mostrar la tabla (equivalente a SELECT * FROM clientes)

In [ ]:
# SQL: SELECT nombre, consumo_kwh FROM clientes
clientes[['nombre', 'consumo_kwh']]

In [ ]:
# SQL: SELECT * FROM clientes WHERE consumo_kwh > 300
clientes[clientes['consumo_kwh'] > 300]

In [ ]:
# SQL: SELECT zona, AVG(consumo_kwh) FROM clientes GROUP BY zona
clientes.groupby('zona')['consumo_kwh'].mean()

**Resumen rápido de equivalencias:**

| SQL | Python (pandas) |
|-----|----------------|
| `SELECT *` | `df` |
| `SELECT col1, col2` | `df[['col1', 'col2']]` |
| `WHERE x > 5` | `df[df['x'] > 5]` |
| `GROUP BY zona` | `df.groupby('zona')` |
| `ORDER BY x DESC` | `df.sort_values('x', ascending=False)` |
| `COUNT(*)` | `len(df)` o `df.shape[0]` |
| `DESCRIBE tabla` | `df.describe()` |
| `SELECT TOP 5` | `df.head()` |

Con esto alcanza. Vamos a lo importante.

---
## Sección 1: El Paisaje de la Inteligencia Artificial

Antes de escribir código, definamos el mapa. No todo es "IA Generativa".

### Los 4 niveles (círculos concéntricos)

```
  ┌─────────────────────────────────────────────┐
  │  Inteligencia Artificial (IA)                │
  │  ┌─────────────────────────────────────┐     │
  │  │  Machine Learning (ML)              │     │
  │  │  ┌─────────────────────────────┐    │     │
  │  │  │  Deep Learning (DL)         │    │     │
  │  │  │  ┌─────────────────────┐    │    │     │
  │  │  │  │  IA Generativa      │    │    │     │
  │  │  │  └─────────────────────┘    │    │     │
  │  │  └─────────────────────────────┘    │     │
  │  └─────────────────────────────────────┘     │
  └─────────────────────────────────────────────┘
```

- **Inteligencia Artificial (IA):** El concepto más amplio. Máquinas que imitan inteligencia humana. Incluye desde reglas IF/ELSE hasta ChatGPT.

- **Machine Learning (ML):** Algoritmos que **aprenden de datos históricos** en lugar de seguir reglas programadas a mano. *Ejemplo en energía: predecir demanda horaria a partir de temperatura y hora del día.*

- **Deep Learning (DL):** Redes neuronales con muchas capas, para datos complejos (imágenes, audio, secuencias largas). *Ejemplo: analizar imágenes termográficas de transformadores para detectar fallas.*

- **IA Generativa (GenAI):** Modelos que **crean contenido nuevo** (texto, imágenes, código). *Ejemplo: un copiloto que resume reportes de incidentes automáticamente.*

### Qué NO es IA

- Un dashboard de Power BI **no es IA** (es visualización).
- Una regla fija tipo `IF consumo > 1000 THEN alerta` **no es ML** (es programación clásica).
- Un modelo de ML **no "piensa"**. Encuentra patrones estadísticos en datos.
- **ML no reemplaza expertos.** Los potencia dándoles herramientas para tomar mejores decisiones.

### En este workshop nos enfocamos en:

| Módulo | Nivel | Ejemplo |
|--------|-------|--------|
| 1 (hoy) | ML - Regresión | Predecir demanda energética |
| 2 | ML - Clasificación + No supervisado | Detectar fraude, segmentar clientes |
| 3 | DL + Ensembles | Series temporales con LSTM, XGBoost |
| 4 | IA Generativa | RAG sobre manuales técnicos, copilotos |

---
## Sección 2: Anatomía de un Modelo de ML

Para que una máquina "aprenda", necesitamos estructurar los datos de una forma específica.

### Conceptos clave

Piensen en una tabla SQL con varias columnas:

```
 Features (X) - lo que usamos para predecir       Target (y) - lo que queremos predecir
 ─────────────────────────────────────────────     ─────────────────────────────────────
  temperatura │  hora  │ dia_semana │ es_finde  │   demanda_MW
 ─────────────┼────────┼────────────┼───────────┤  ───────────
     32.5     │   14   │     lunes  │   False   │     420
     28.1     │   03   │   domingo  │   True    │     310
     15.2     │   19   │   miercol  │   False   │     380
```

- **Features (X):** Las columnas que usamos como "pistas".
- **Target (y):** Lo que queremos predecir. La columna de "respuesta".
- **Train/Test Split:** Separamos los datos en dos grupos:
  - **Entrenamiento (~80%):** Los datos con los que el modelo "estudia".
  - **Test (~20%):** Los datos del "examen final". El modelo NUNCA los vio antes.

### Leakage (fuga de datos)

Un fallo crítico de diseño. Es como darle las respuestas del examen al alumno durante el estudio. El modelo parece genial, pero en la realidad falla.

**Ejemplo concreto en energía:** Supongamos que queremos predecir la demanda de las 15:00hs. Si usamos como feature la "demanda de las 14:00hs", en el entrenamiento funciona perfecto (el dato ya existe en la tabla). Pero en producción, cuando son las 14:30 y necesitamos predecir las 15:00, esa lectura todavía no está consolidada en el sistema. Resultado: el modelo funciona en el laboratorio pero falla en el mundo real.

**Regla de oro:** Solo usar features que van a estar disponibles en el momento real de hacer la predicción.

### Overfitting vs Underfitting

Dos formas comunes en que un modelo puede fallar:

| Concepto | Qué pasa | Analogía |
|----------|----------|----------|
| **Underfitting** | El modelo es demasiado simple para capturar el patrón | Un alumno que solo leyó el título del libro |
| **Overfitting** | El modelo memoriza los datos de entrenamiento, incluyendo el ruido | Un alumno que memorizó las respuestas del parcial anterior pero no entiende el tema |
| **Buen ajuste** | El modelo captura el patrón real sin memorizar ruido | Un alumno que entiende los conceptos y puede resolver problemas nuevos |

Vamos a ver estos conceptos en acción más adelante con un ejemplo práctico.

In [ ]:
# Ejemplo: asi se ve la estructura de datos para ML
ejemplo = pd.DataFrame({
    'temperatura': [32.5, 28.1, 15.2, 5.0, 38.0],
    'hora': [14, 3, 19, 8, 15],
    'es_finde': [False, True, False, False, True],
    'demanda_MW': [420, 310, 380, 390, 480]
})

print("Features (X) - las pistas:")
print(ejemplo[['temperatura', 'hora', 'es_finde']])
print("\nTarget (y) - lo que queremos predecir:")
print(ejemplo['demanda_MW'])

### Tipos de problemas en ML

| Tipo | Target | Ejemplo en energía |
|------|--------|-------------------|
| **Regresión** | Un número continuo | Predecir MW de demanda |
| **Clasificación** | Una categoría (Sí/No, A/B/C) | Detectar fraude (Sí/No) |
| **Clustering** | No hay target (no supervisado) | Agrupar clientes similares |

Hoy nos enfocamos en **Regresión**.

---
## Sección 3: Caso Práctico - Predicción de Demanda Energética

**Objetivo:** Construir un modelo que prediga la demanda de energía (en MW) basándose en temperatura, hora del día y día de la semana.

**Métrica de éxito:** MAE (Error Absoluto Medio) - en promedio, ¿por cuántos MW nos equivocamos cada hora?

### 3.0 Cargar los datos

Vamos a trabajar con datos sintéticos que simulan un año de operación de una distribuidora. Los datos tienen:
- **Registros horarios** de enero a diciembre 2024
- **Temperatura** que varía por estación y hora del día
- **Demanda en MW** que depende de la temperatura (más frío o más calor = más consumo) y la hora

In [ ]:
# Generamos datos sintéticos que simulan un año de operación
np.random.seed(42)
fechas = pd.date_range("2024-01-01", "2024-12-31 23:00:00", freq="h")
n = len(fechas)

# Temperatura: varía por estación del año y hora del día
dia_del_anio = fechas.dayofyear
hora = fechas.hour
temperatura = (20
    - 12 * np.cos(2 * np.pi * dia_del_anio / 365)  # Estacional
    + 4 * np.sin(2 * np.pi * hora / 24)            # Ciclo diario
    + np.random.normal(0, 2, n))                    # Variabilidad

# Demanda: relación en U con temperatura + ciclo diario
# (más frío O más calor -> más demanda por calefacción/aire acondicionado)
demanda = (300
    + 0.8 * (temperatura - 20)**2       # Efecto U de temperatura
    + 8 * np.sin(2 * np.pi * (hora - 6) / 24)  # Ciclo diario
    + np.random.normal(0, 5, n))        # Ruido

df = pd.DataFrame({
    'fecha': fechas,
    'temperatura': np.round(temperatura, 1),
    'hora': hora,
    'dia_semana': fechas.dayofweek,
    'es_finde': (fechas.dayofweek >= 5).astype(int),
    'demanda_MW': np.round(demanda, 1)
})

print(f"Dataset: {df.shape[0]} registros horarios, {df.shape[1]} columnas")
print(f"Período: {df['fecha'].min()} a {df['fecha'].max()}")

In [ ]:
# Equivalente a SELECT TOP 5 * FROM datos
df.head()

In [ ]:
# Equivalente a un DESCRIBE de la tabla
df.describe()

### 3.1 Exploración visual

Antes de modelar, siempre hay que **mirar los datos**. Tres gráficos clave:

In [ ]:
# Gráfico 1: Demanda durante la primera semana de enero
plt.figure(figsize=(12, 4))
semana1 = df[df['fecha'] < '2024-01-08']
plt.plot(semana1['fecha'], semana1['demanda_MW'])
plt.title('Demanda Energética - Primera Semana de Enero')
plt.xlabel('Fecha')
plt.ylabel('MW')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 2: Relación Temperatura vs Demanda (CLAVE - notar la forma de U)
plt.figure(figsize=(8, 5))
muestra = df.sample(2000, random_state=42)  # Tomamos una muestra para que se vea mejor
plt.scatter(muestra['temperatura'], muestra['demanda_MW'], alpha=0.3, s=10)
plt.title('Temperatura vs Demanda: notar la forma de U')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Demanda (MW)')
plt.show()
print("Mucho frío -> más demanda (calefacción)")
print("Mucho calor -> más demanda (aire acondicionado)")
print("Temperatura agradable (~20°C) -> demanda mínima")

In [ ]:
# Gráfico 3: Demanda promedio por hora del día
demanda_por_hora = df.groupby('hora')['demanda_MW'].mean()
plt.figure(figsize=(10, 4))
plt.plot(demanda_por_hora.index, demanda_por_hora.values, color='steelblue')
plt.title('Demanda Promedio por Hora del Día')
plt.xlabel('Hora')
plt.ylabel('Demanda Promedio (MW)')
plt.xticks(range(24))
plt.show()

### Correlaciones entre variables

Antes de modelar, veamos qué tan **correlacionadas** están las variables. La correlación mide la fuerza de la relación **lineal** entre dos variables (de -1 a +1). Pero atención: no detecta relaciones no lineales.

In [ ]:
# Matriz de correlación
plt.figure(figsize=(8, 6))
cols_corr = ['temperatura', 'hora', 'dia_semana', 'es_finde', 'demanda_MW']
correlacion = df[cols_corr].corr()
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Matriz de Correlación entre Variables')
plt.tight_layout()
plt.show()

print("¡Atención: la correlación de temperatura con demanda es casi 0!")
print("Esto NO significa que no estén relacionadas.")
print("Significa que la relación NO es lineal (es una U).")
print("Lección: la correlación solo detecta relaciones lineales. No confiar ciegamente en ella.")

### ¿Qué aprendimos de la exploración?

Antes de pasar a construir modelos, resumamos las observaciones clave de los gráficos:

1. **La demanda tiene un ciclo diario claro** — sube durante el día y baja de noche (gráfico 1 y 3).
2. **La relación con la temperatura tiene forma de U** — tanto el frío como el calor aumentan la demanda (gráfico 2). Esto es importante porque muchos modelos simples asumen relaciones lineales.
3. **La correlación lineal entre temperatura y demanda es casi cero** — pero eso no significa que no estén relacionadas. La correlación solo mide relaciones en línea recta, y la nuestra es una curva.

Estas observaciones van a ser fundamentales cuando elijamos qué modelos usar y qué features crear. Ahora sí, pasemos a modelar.

### 3.2 Dividir datos: Entrenamiento vs Prueba

No podemos evaluar el modelo con los mismos datos que usó para aprender. Sería como evaluar a un alumno con las mismas preguntas que estudió.

**¿Por qué cortamos cronológicamente?** En muchos tutoriales de ML se mezclan los datos al azar antes de separar train/test. Pero con datos de energía (o cualquier serie temporal) **eso es trampa**: estaríamos usando datos del futuro para predecir el pasado. En la realidad, cuando hacemos una predicción, solo tenemos datos del pasado disponibles.

Por eso cortamos en un punto del tiempo: entrenamos con Ene-Oct y probamos con Nov-Dic, simulando lo que pasaría si pusiéramos el modelo en producción en noviembre.

In [ ]:
# Corte cronológico: 80% entrenamiento, 20% prueba
corte = int(len(df) * 0.8)
train = df.iloc[:corte]
test = df.iloc[corte:]

# Definimos Features (X) y Target (y)
features = ['temperatura', 'hora', 'es_finde']
target = 'demanda_MW'

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print(f"Entrenamiento: {len(X_train)} registros (hasta {train['fecha'].max().date()})")
print(f"Prueba: {len(X_test)} registros (desde {test['fecha'].min().date()})")

### 3.3 Baseline Ingenuo: el piso mínimo

Antes de usar cualquier modelo, preguntemos: **¿qué tan bien lo haríamos prediciendo siempre el promedio histórico?**

Si nuestro modelo "inteligente" no supera esto, no sirve para nada.

**MAE (Mean Absolute Error)** = En promedio, ¿por cuántos MW nos equivocamos?

In [ ]:
from sklearn.metrics import mean_absolute_error

# Predicción ingenua: siempre predecir el promedio del entrenamiento
promedio_train = y_train.mean()
pred_baseline = [promedio_train] * len(y_test)

mae_baseline = mean_absolute_error(y_test, pred_baseline)
print(f"Promedio histórico de demanda: {promedio_train:.1f} MW")
print(f"MAE del Baseline (predecir siempre el promedio): {mae_baseline:.2f} MW")
print(f"\nInterpretación: si solo usamos el promedio, erramos por ~{mae_baseline:.0f} MW cada hora.")
print(f"Para dimensionar: {mae_baseline:.0f} MW es aproximadamente el consumo de {mae_baseline/3.5:.0f} hogares.")
print(f"Nuestro objetivo es bajar ese error lo más posible con modelos más inteligentes.")

In [ ]:
# Visualización: ¿qué tan malo es el baseline?
plt.figure(figsize=(14, 5))
rango_vis = 200  # Primeras 200 horas del período de prueba
horas_vis = range(rango_vis)
plt.plot(horas_vis, y_test.values[:rango_vis], color='black', linewidth=1.2,
         label='Demanda real', alpha=0.9)
plt.axhline(y=promedio_train, color='#d9534f', linewidth=2.5, linestyle='--',
            label=f'Baseline: siempre {promedio_train:.0f} MW')
plt.fill_between(horas_vis, promedio_train, y_test.values[:rango_vis],
                 alpha=0.15, color='red')
plt.title('Baseline vs Realidad: predecir siempre el promedio histórico', fontsize=13)
plt.xlabel('Hora (período de prueba)')
plt.ylabel('Demanda (MW)')
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

print("El área sombreada es el error: la diferencia entre la realidad y nuestra predicción.")
print("Predecir el promedio no captura ningún patrón. Es nuestro piso mínimo a superar.")

### 3.4 Primer modelo: Regresión Lineal

La regresión lineal busca una **línea recta** (o un plano) que mejor se ajuste a los datos.

Fórmula: `demanda = a*temperatura + b*hora + c*es_finde + d`

**¿Qué significa "entrenar" un modelo?** Cuando ejecutamos `.fit()`, el algoritmo analiza los datos de entrenamiento y busca los mejores valores para los coeficientes `a`, `b`, `c` y `d` — los que hagan que la fórmula se equivoque lo menos posible. Una vez que encontró esos valores, con `.predict()` podemos aplicar la fórmula a datos nuevos que el modelo nunca vio.

Veamos qué pasa...

In [ ]:
from sklearn.linear_model import LinearRegression

# Entrenar el modelo
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

# Predecir
pred_lineal = modelo_lineal.predict(X_test)

# Evaluar
mae_lineal = mean_absolute_error(y_test, pred_lineal)
print(f"MAE Regresion Lineal: {mae_lineal:.2f} MW")
print(f"MAE Baseline:         {mae_baseline:.2f} MW")
print(f"\nDiferencia: {mae_baseline - mae_lineal:.2f} MW")

In [ ]:
# ¿Qué "aprendió" el modelo lineal? Miremos sus coeficientes
print("Coeficientes de la Regresión Lineal:")
print("="*50)
for feat, coef in zip(features, modelo_lineal.coef_):
    direccion = "↑" if coef > 0 else "↓"
    print(f"  {feat:15s}: {coef:+8.4f}  ({direccion} por cada unidad que sube)")
print(f"  {'intercepto':15s}: {modelo_lineal.intercept_:+8.4f}")

print(f"\nAnálisis:")
print(f"  El coeficiente de temperatura ({modelo_lineal.coef_[0]:+.4f}) es muy chico.")
print(f"  El modelo 'cree' que la temperatura casi no importa.")
print(f"  Pero sabemos que SÍ importa, solo que de forma NO LINEAL (la U).")
print(f"  Una recta no puede capturar que tanto frío como calor aumentan la demanda.")

### Sorpresa: ¡la regresión lineal funciona MAL!

El modelo lineal **es peor que el baseline**. ¿Por qué?

Miremos el scatter plot de nuevo. La relación temperatura-demanda es una **curva en U**, no una línea recta. La regresión lineal no puede capturar eso.

In [ ]:
# Visualizar el problema: la regresión lineal intenta trazar una recta
# pero la relación es una curva en U
plt.figure(figsize=(10, 5))
plt.scatter(muestra['temperatura'], muestra['demanda_MW'], alpha=0.2, s=10, label='Datos reales')

# Mostrar la "línea" que traza la regresión lineal
temps_ordenadas = np.linspace(df['temperatura'].min(), df['temperatura'].max(), 100)
preds_linea = modelo_lineal.predict(pd.DataFrame({
    'temperatura': temps_ordenadas,
    'hora': [12]*100,
    'es_finde': [0]*100
}))
plt.plot(temps_ordenadas, preds_linea, 'r-', linewidth=3, label='Regresión Lineal')

plt.title('El problema: la línea recta no captura la curva en U')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Demanda (MW)')
plt.legend()
plt.show()
print("La línea sube para temperaturas altas, pero NO sube para temperaturas bajas.")
print("Necesitamos un modelo que entienda la forma de U.")

### ¿Qué curva se ajusta mejor?

La recta no funciona porque la relación es una **curva en U**. Pero entonces, ¿qué tipo de curva necesitamos? Comparemos distintos enfoques para entender mejor el problema:

In [ ]:
# ¿Qué curva se ajusta mejor? Comparemos distintos intentos:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('¿Qué curva se ajusta mejor a los datos?',
             fontsize=14, fontweight='bold')

# Datos para graficar
muestra_reg = df.sample(1500, random_state=42)
temps = muestra_reg['temperatura'].values
dem = muestra_reg['demanda_MW'].values
t_sorted = np.linspace(temps.min() + 1, temps.max() - 1, 300)

# --- Intento 1: Promedio (baseline) ---
ax = axes[0, 0]
ax.scatter(temps, dem, alpha=0.12, s=6, color='gray')
ax.axhline(y=promedio_train, color='#d9534f', linewidth=3)
ax.set_title('Intento 1: El promedio', fontsize=11, fontweight='bold', color='#d9534f')
ax.set_ylabel('Demanda (MW)')
ax.text(0.95, 0.05, 'Solo una línea plana.\nIgnora la temperatura.',
        transform=ax.transAxes, ha='right', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Intento 2: Recta (regresión lineal) ---
ax = axes[0, 1]
ax.scatter(temps, dem, alpha=0.12, s=6, color='gray')
z1 = np.polyfit(temps, dem, 1)
ax.plot(t_sorted, np.poly1d(z1)(t_sorted), color='#f0ad4e', linewidth=3)
ax.set_title('Intento 2: Una recta', fontsize=11, fontweight='bold', color='#f0ad4e')
ax.text(0.95, 0.05, 'Mejor, pero no puede\nrepresentar la forma de U.',
        transform=ax.transAxes, ha='right', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Intento 3: Parábola (cuadrática) ---
ax = axes[1, 0]
ax.scatter(temps, dem, alpha=0.12, s=6, color='gray')
z2 = np.polyfit(temps, dem, 2)
ax.plot(t_sorted, np.poly1d(z2)(t_sorted), color='#5cb85c', linewidth=3)
ax.set_title('Intento 3: Una parábola (U)', fontsize=11, fontweight='bold', color='#5cb85c')
ax.set_xlabel('Temperatura (°C)')
ax.set_ylabel('Demanda (MW)')
ax.text(0.95, 0.05, '¡Captura la U!\nFrío y calor = más demanda.',
        transform=ax.transAxes, ha='right', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# --- Intento 4: Polinomio de grado MUY alto (overfitting) ---
ax = axes[1, 1]
ax.scatter(temps, dem, alpha=0.12, s=6, color='gray')
# Muestra chica para que el overfitting sea visible
np.random.seed(7)
idx_small = np.random.choice(len(temps), 40, replace=False)
t_small = np.sort(temps[idx_small])
d_small = dem[idx_small][np.argsort(temps[idx_small])]
ax.scatter(t_small, d_small, color='#8e44ad', s=40, zorder=5,
           edgecolors='black', linewidth=0.5)
z_high = np.polyfit(t_small, d_small, 12)
t_fine = np.linspace(t_small[1], t_small[-2], 500)
y_high = np.clip(np.poly1d(z_high)(t_fine), dem.min() - 30, dem.max() + 30)
ax.plot(t_fine, y_high, color='#8e44ad', linewidth=2.5)
ax.set_ylim(dem.min() - 20, dem.max() + 20)
ax.set_title('Intento 4: Curva MUY compleja', fontsize=11, fontweight='bold', color='#8e44ad')
ax.set_xlabel('Temperatura (°C)')
ax.text(0.95, 0.05, 'Se retuerce para pasar por\ncada punto. ¡Eso es OVERFITTING!',
        transform=ax.transAxes, ha='right', fontsize=9, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("RESUMEN:")
print("  Intento 1 (Promedio):    Demasiado simple → UNDERFITTING")
print("  Intento 2 (Recta):       No captura la U  → UNDERFITTING")
print("  Intento 3 (Parábola U):  Captura el patrón real → BUEN AJUSTE")
print("  Intento 4 (Grado 12):    Memoriza ruido   → OVERFITTING")
print("\nEl arte de ML: encontrar el punto justo entre simple y complejo.")

### 3.5 Feature Engineering: darle pistas al modelo

**La lección más importante del workshop:** entender los datos es más importante que elegir el algoritmo.

Sabemos dos cosas del negocio:
1. La relación temperatura-demanda es cuadrática (U). Si le damos al modelo `(temperatura - 20)²`, captura esa forma directamente.
2. La demanda varía por estación del año. Agregar el **mes** como feature le da información estacional.

Esto se llama **Feature Engineering** (ingeniería de características): crear columnas nuevas a partir del conocimiento del negocio.

In [ ]:
# Agregar features con conocimiento del negocio
df['temp_cuadrada'] = (df['temperatura'] - 20)**2  # Captura la relación en U
df['mes'] = df['fecha'].dt.month                     # Captura estacionalidad

# Recalcular train/test con las nuevas features
features_v2 = ['temperatura', 'temp_cuadrada', 'hora', 'es_finde', 'mes']

X_train_v2 = df.iloc[:corte][features_v2]
X_test_v2 = df.iloc[corte:][features_v2]

# Entrenar modelo lineal con las nuevas features
modelo_lineal_v2 = LinearRegression()
modelo_lineal_v2.fit(X_train_v2, y_train)
pred_lineal_v2 = modelo_lineal_v2.predict(X_test_v2)

mae_lineal_v2 = mean_absolute_error(y_test, pred_lineal_v2)
print(f"MAE Baseline:                     {mae_baseline:.2f} MW")
print(f"MAE Reg. Lineal (sin feat. eng.):  {mae_lineal:.2f} MW")
print(f"MAE Reg. Lineal (con feat. eng.):  {mae_lineal_v2:.2f} MW  <-- ¡MEJORA ENORME!")

**Con dos columnas nuevas (`temp_cuadrada` y `mes`), el modelo mejoró dramáticamente!**

Esto es Feature Engineering: usar conocimiento del negocio para ayudar al modelo. Un data scientist que entiende el dominio de energía va a crear mejores features que uno que solo conoce algoritmos.

### 3.6 Árbol de Decisión

El árbol de decisión divide los datos con preguntas tipo: *"¿La temperatura es mayor a 25? Sí -> rama derecha, No -> rama izquierda"*.

La ventaja: captura relaciones no lineales **sin necesidad de feature engineering manual**.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Usamos las features originales (sin temp_cuadrada) - el arbol no la necesita
modelo_arbol = DecisionTreeRegressor(max_depth=5, random_state=42)
modelo_arbol.fit(X_train, y_train)
pred_arbol = modelo_arbol.predict(X_test)

mae_arbol = mean_absolute_error(y_test, pred_arbol)
print(f"MAE Arbol de Decision: {mae_arbol:.2f} MW")

In [ ]:
# Visualizar el árbol de decisión: cada nodo es una "pregunta"
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(modelo_arbol,
          feature_names=features,
          filled=True,
          rounded=True,
          fontsize=9,
          max_depth=3,  # Solo primeros 3 niveles para que sea legible
          impurity=False)
plt.title('Árbol de Decisión (primeros 3 niveles)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Cómo leer el árbol:")
print("  - Cada caja hace una pregunta (ej: 'temperatura <= 25.3?')")
print("  - SÍ → rama izquierda, NO → rama derecha")
print("  - 'value': predicción promedio de demanda en ese grupo")
print("  - 'samples': cuántos registros llegan a ese nodo")
print("  - Color más oscuro = demanda más alta")
print("\nA diferencia de la regresión lineal, el árbol puede hacer preguntas")
print("como '¿la temperatura está entre 15 y 25?' y dar una predicción distinta")
print("para cada rango. Por eso captura la forma de U naturalmente.")

### Overfitting en acción

En la Sección 2 introdujimos el concepto de overfitting vs underfitting. Ahora veámoslo con datos reales.

Vamos a entrenar árboles de decisión con distinta **profundidad** (complejidad) y medir el error tanto en los datos de entrenamiento como en los de test:

In [ ]:
# Demostración práctica: Overfitting vs Underfitting
# Entrenamos árboles con distinta profundidad y medimos el error

depths = list(range(1, 25))
mae_train_list = []
mae_test_list = []

for d in depths:
    tree_temp = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree_temp.fit(X_train, y_train)
    mae_train_list.append(mean_absolute_error(y_train, tree_temp.predict(X_train)))
    mae_test_list.append(mean_absolute_error(y_test, tree_temp.predict(X_test)))

best_depth = depths[np.argmin(mae_test_list)]

plt.figure(figsize=(12, 5))
plt.plot(depths, mae_train_list, 'o-', label='Error en Entrenamiento', color='steelblue', linewidth=2)
plt.plot(depths, mae_test_list, 'o-', label='Error en Test (datos nuevos)', color='#d9534f', linewidth=2)
plt.axvline(x=5, color='green', linestyle='--', alpha=0.7, linewidth=2,
            label=f'Nuestro modelo (max_depth=5)')
plt.axvline(x=best_depth, color='orange', linestyle=':', alpha=0.7, linewidth=2,
            label=f'Óptimo (max_depth={best_depth})')

# Zonas de underfitting y overfitting
plt.axvspan(0.5, 3.5, alpha=0.06, color='blue')
plt.axvspan(12.5, 24.5, alpha=0.06, color='red')
plt.text(2, max(mae_test_list)*0.95, 'UNDERFITTING\n(muy simple)', ha='center',
         fontsize=10, color='blue', style='italic')
plt.text(19, max(mae_test_list)*0.95, 'OVERFITTING\n(muy complejo)', ha='center',
         fontsize=10, color='red', style='italic')

plt.xlabel('Profundidad máxima del árbol (complejidad)', fontsize=11)
plt.ylabel('MAE (MW)', fontsize=11)
plt.title('Overfitting: cuando el modelo memoriza en vez de aprender', fontsize=13)
plt.legend(loc='center right')
plt.tight_layout()
plt.show()

print("Observar:")
print("  - Error de entrenamiento (azul): SIEMPRE baja → el árbol memoriza cada vez más")
print("  - Error de test (rojo): baja al principio, luego SUBE → deja de generalizar")
print(f"  - Punto óptimo: max_depth={best_depth} (error test mínimo: {min(mae_test_list):.2f} MW)")
print(f"  - Nuestro max_depth=5 está cerca del óptimo")
print(f"\nEsta es la razón por la que siempre evaluamos con datos que el modelo NO vio (test set).")

### 3.7 Comparación final de modelos

In [ ]:
from sklearn.metrics import r2_score

# R²: 1.0 = perfecto, 0.0 = igual que el promedio, <0 = peor que el promedio
r2_baseline = r2_score(y_test, pred_baseline)
r2_lineal = r2_score(y_test, pred_lineal)
r2_lineal_v2 = r2_score(y_test, pred_lineal_v2)
r2_arbol = r2_score(y_test, pred_arbol)

# Tabla resumen
resultados = pd.DataFrame({
    'Modelo': ['Baseline (promedio)', 'Reg. Lineal (básica)', 'Reg. Lineal + Feat. Eng.', 'Árbol de Decisión'],
    'MAE (MW)': [mae_baseline, mae_lineal, mae_lineal_v2, mae_arbol],
    'R²': [r2_baseline, r2_lineal, r2_lineal_v2, r2_arbol]
}).sort_values('MAE (MW)')

print("Comparación de Modelos:")
print("="*70)
print(f"  {'Modelo':30s} {'MAE (MW)':>10s} {'R²':>10s}")
print("-"*70)
for _, row in resultados.iterrows():
    print(f"  {row['Modelo']:30s} {row['MAE (MW)']:10.2f} {row['R²']:10.4f}")
print()
print("MAE = Error Absoluto Medio (menor es mejor)")
print("R²  = Coeficiente de determinación (más cercano a 1 es mejor)")
print("      R² = 0 significa 'igual que predecir el promedio'")
print("      R² < 0 significa 'peor que predecir el promedio'")

In [ ]:
# Gráfico de barras
plt.figure(figsize=(10, 5))
colores = [ '#5cb85c', '#5cb85c', '#d9534f', '#d9534f']
plt.barh(resultados['Modelo'], resultados['MAE (MW)'], color=colores)
plt.xlabel('MAE (MW) - menor es mejor')
plt.title('Comparación de Modelos')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: predicciones vs realidad (primeras 100 horas del test)
plt.figure(figsize=(14, 5))
rango = 100
plt.plot(y_test.values[:rango], label='Realidad', color='black', linewidth=2)
plt.plot(pred_lineal_v2[:rango], label='Reg. Lineal + Feat. Eng.', linestyle='--', alpha=0.8)
plt.plot(pred_arbol[:rango], label='Árbol de Decisión', linestyle='--', alpha=0.8)
plt.title('Predicciones vs Realidad (primeras 100 horas de prueba)')
plt.ylabel('Demanda (MW)')
plt.xlabel('Hora')
plt.legend()
plt.show()

### Lecciones clave de Regresión

1. **Siempre arrancar con un baseline simple.** Si no lo superás, tu modelo no aporta valor.
2. **Entender los datos > elegir el algoritmo.** El feature engineering (temp²) mejoró más que cambiar de modelo.
3. **Modelos simples pueden fallar** si la relación no es lineal (curva en U).
4. **Árboles de decisión** capturan relaciones no lineales automáticamente.

---
## Sección 4: Teaser - Problemas de Clasificación

Hasta ahora predijimos un **número** (demanda en MW). Eso es **Regresión**.

Pero ¿qué pasa si en vez de un número queremos predecir una **categoría**? Por ejemplo:
- ¿Este patrón de consumo es **Normal** o **Anómalo**?
- ¿Esta lectura de medidor indica **fraude** o es **legítima**?
- ¿Este transformador va a **fallar** en los próximos 30 días?

En todos estos casos la respuesta no es un número, sino una **etiqueta**: Sí/No, Normal/Anómalo, Falla/No falla. Esto se llama **Clasificación**.

### ¿En qué se diferencia de la regresión?

| | Regresión | Clasificación |
|--|-----------|---------------|
| **¿Qué predice?** | Un número continuo | Una categoría |
| **Ejemplo** | "La demanda será 420 MW" | "Este consumo es anómalo" |
| **Métrica principal** | MAE (error en MW) | Accuracy, Precision, Recall |
| **Algoritmo usado** | `DecisionTreeRegressor` | `DecisionTreeClassifier` |

La mecánica es muy similar a lo que ya hicimos: tenemos features (X), un target (y), dividimos en train/test y entrenamos un modelo. Lo que cambia es **qué tipo de respuesta da el modelo** y **cómo medimos si lo hace bien**.

Veamos un ejemplo rápido con nuestros datos de energía.

### 4.1 Crear los datos de clasificación

Para este ejemplo, vamos a crear una etiqueta de **anomalía** a partir de los datos que ya tenemos. La idea es simple:

1. Sabemos cuál es el patrón "normal" de consumo (depende de la temperatura y la hora).
2. Calculamos cuánto se desvía el consumo real de ese patrón esperado.
3. Si el desvío es **muy grande** (más de 12 MW), lo marcamos como **anomalía**.

En el mundo real, estas etiquetas podrían venir de inspecciones de campo, reportes de técnicos, o alertas históricas del sistema SCADA. Acá las fabricamos nosotros para practicar.

In [ ]:
# Calculamos cuánto se desvió el consumo real del patrón esperado
consumo_esperado = 300 + 0.8 * (df['temperatura'] - 20)**2 + 8 * np.sin(2 * np.pi * (df['hora'] - 6) / 24)
residuo = df['demanda_MW'] - consumo_esperado

# Si el desvío es mayor a 12 MW -> anomalía
df['anomalia'] = (np.abs(residuo) > 12).astype(int)

print(f"Distribución de clases:")
print(f"  Normal (0):   {(df['anomalia']==0).sum()} ({(df['anomalia']==0).mean():.1%})")
print(f"  Anomalía (1): {(df['anomalia']==1).sum()} ({(df['anomalia']==1).mean():.1%})")

Notar que la gran mayoría de registros son **normales** y solo un porcentaje chico son anomalías. Esto es típico en problemas reales: los eventos "interesantes" (fallas, fraudes, anomalías) son raros.

Veamos cómo se ven las anomalías en los datos:

In [ ]:
# Visualizar: ¿dónde están las anomalías?
plt.figure(figsize=(10, 5))
normales = df[df['anomalia'] == 0].sample(2000, random_state=42)
anomalos = df[df['anomalia'] == 1].sample(min(500, df['anomalia'].sum()), random_state=42)

plt.scatter(normales['temperatura'], normales['demanda_MW'],
            alpha=0.15, s=8, color='steelblue', label='Normal')
plt.scatter(anomalos['temperatura'], anomalos['demanda_MW'],
            alpha=0.5, s=15, color='red', label='Anomalía')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Demanda (MW)')
plt.title('Consumos normales vs anómalos')
plt.legend()
plt.tight_layout()
plt.show()

print("Las anomalías (rojo) son los puntos que se alejan del patrón esperado.")
print("El modelo de clasificación debe aprender a distinguir rojos de azules.")

### 4.2 Entrenar un clasificador

Vamos a usar un **Árbol de Decisión** — el mismo tipo de algoritmo que usamos para regresión — pero ahora configurado para **clasificación** (`DecisionTreeClassifier` en vez de `DecisionTreeRegressor`).

La diferencia clave: en vez de predecir un número (ej: "420 MW"), el modelo predice una **clase**: 0 (Normal) o 1 (Anomalía).

Usamos las mismas features de antes (temperatura, hora, fin de semana) y medimos con **accuracy**: el porcentaje de predicciones correctas sobre el total.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Preparar datos
features_clas = ['temperatura', 'hora', 'es_finde']
X_train_c = df.iloc[:corte][features_clas]
X_test_c = df.iloc[corte:][features_clas]
y_train_c = df.iloc[:corte]['anomalia']
y_test_c = df.iloc[corte:]['anomalia']

# Entrenar clasificador
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train_c, y_train_c)
pred_clas = clf.predict(X_test_c)

print(f"Accuracy: {accuracy_score(y_test_c, pred_clas):.1%}")

### 4.3 Matriz de Confusión: ¿dónde se equivoca el modelo?

El accuracy nos da un solo número ("acertó el X%"), pero no nos dice **en qué tipo de errores** se equivoca. Para eso usamos la **Matriz de Confusión**: una tabla de 2x2 que cruza lo que el modelo predijo contra lo que realmente pasó.

```
                        Predicción del Modelo
                        Normal       Anomalía
              Normal  │ ✅ OK        │ ⚠️ Falsa alarma   │
  Realidad            │              │                    │
              Anomalía│ ❌ Se escapó  │ ✅ Detectada       │
```

Hay 4 resultados posibles:
- **Verdadero Negativo (✅):** Era normal y el modelo dijo normal → perfecto.
- **Falso Positivo (⚠️):** Era normal pero el modelo dijo anomalía → mandamos un técnico innecesariamente.
- **Falso Negativo (❌):** Era anomalía pero el modelo dijo normal → ¡se nos escapó una falla!
- **Verdadero Positivo (✅):** Era anomalía y el modelo la detectó → perfecto.

En energía, los **Falsos Negativos** suelen ser más graves: no detectar una falla puede causar daños en equipos o cortes de servicio. Los **Falsos Positivos** generan costos de inspecciones innecesarias, pero son menos peligrosos.

In [ ]:
# Matriz de Confusión: la herramienta clave para clasificación
cm = confusion_matrix(y_test_c, pred_clas)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Normal', 'Pred: Anomalía'],
            yticklabels=['Real: Normal', 'Real: Anomalía'])
plt.title('Matriz de Confusión')
plt.ylabel('Valor Real')
plt.xlabel('Predicción del Modelo')
plt.show()

print("Cómo leer la matriz:")
print(f"  Verdaderos Negativos (Normal bien detectado):   {cm[0][0]}")
print(f"  Falsos Positivos (Falsa alarma):                {cm[0][1]}")
print(f"  Falsos Negativos (Anomalía que se nos escapó):  {cm[1][0]}")
print(f"  Verdaderos Positivos (Anomalía detectada):      {cm[1][1]}")

### 4.4 Métricas más allá del accuracy

La matriz de confusión nos da el panorama completo, pero a veces necesitamos resumirlo en un número. Para eso existen **precision**, **recall** y **F1-score**. 

Veamos el reporte completo y después lo interpretamos:

In [ ]:
# Reporte completo de clasificación
print(classification_report(y_test_c, pred_clas, target_names=['Normal', 'Anomalía']))

### Lectura rápida de métricas de clasificación

| Métrica | Significado | Analogía en energía |
|---------|-------------|--------------------|
| **Precision** | De los que marqué como anomalía, ¿cuántos realmente lo eran? | "¿Mandamos inspectores y realmente había fraude?" |
| **Recall** | De todas las anomalías reales, ¿cuántas detecté? | "¿Se nos escapó algún fraude?" |
| **F1** | Balance entre precision y recall | Compromiso entre falsas alarmas y fraudes no detectados |

**En el próximo módulo profundizamos en clasificación, métricas y datos desbalanceados.**

### El peligro de las clases desbalanceadas

Observen la distribución de nuestro dataset: la gran mayoría de registros son "Normal" y pocos son "Anomalía". Esto se llama **desbalance de clases** y es MUY común en problemas reales de energía (las fallas son raras, el fraude es infrecuente).

**El problema:** Un modelo que SIEMPRE predice "Normal" tendría un accuracy altísimo (acierta en la clase mayoritaria), pero sería completamente inútil para detectar anomalías — que es justamente lo que nos interesa.

**Por eso el accuracy solo no alcanza.** Necesitamos mirar precision, recall y F1. En el **Módulo 2** vamos a profundizar en cómo manejar datos desbalanceados y elegir la métrica correcta según el problema de negocio.

---
## Conclusión del Módulo 1

### Qué aprendimos hoy:

1. **Python básico con pandas** - DataFrames son tablas, como en SQL.
2. **El paisaje de la IA** - IA > ML > DL > GenAI. No todo es ChatGPT.
3. **Anatomía de un modelo** - Features, Target, Train/Test Split.
4. **Regresión** - Predecir un número. Baseline -> Regresión Lineal -> Feature Engineering -> Árbol de Decisión.
5. **Feature Engineering** - Entender los datos importa más que el algoritmo.
6. **Teaser de Clasificación** - Predecir categorías. Matriz de confusión.

### Próximo módulo:

**Módulo 2: Clasificación completa, Clustering (agrupar clientes/transformadores sin etiquetas) y Ciclo de Vida de un proyecto de ML (CRISP-DM).**